## Paso 3. Datos para la simulación del transporte: GWT

Para resolver los cuatro casos definimos la función `get_mst_dict()` que genera la información necesaria para el paquete MST. 


In [ ]:
def get_mst_dict(phys):
    mst = dict(
        porosity = phys["porosity"],
        sorption = None,
        bulk_density = None,
        distcoef = None, 
        first_order_decay = None,
        decay = None,
        decay_sorbed = None
    )
    
    if phys["retardation_factor"] > 1.0:
        mst["sorption"] = "linear"
        mst["bulk_density"] = 1.0
        mst["distcoef"] = (phys["retardation_factor"] - 1.0) * phys["porosity"] / mst["bulk_density"]

    if phys["decay_rate"] != 0.0:
        mst["first_order_decay"] = True
        mst["decay"] = phys["decay_rate"]
        if phys["retardation_factor"] > 1.0:
            mst["decay_sorbed"] = phys["decay_rate"]

    return mst

### Determinación de cada caso

In [ ]:
long_disp = [0.1, 1.0, 1.0, 1.0]
reta_fact = [1.0, 1.0, 2.0, 1.0]
deca_rate = [0.0, 0.0, 0.0, 0.01]
dir_names = ['p01a','p01b','p01c','p01d']

case = 1 #int(input("Caso (1-4)= ")) - 1 # 0, 1, 2, 3
dirname = dir_names[case]

# Agregamos más parámetros físicos al diccionario phys
phys["longitudinal_dispersivity"] = long_disp[case]
phys["retardation_factor"] = reta_fact[case]
phys["decay_rate"] = deca_rate[case]
phys["dispersion_coefficient"] = phys["longitudinal_dispersivity"] * phys["specific_discharge"] / phys["retardation_factor"]

xmf6.nice_print(phys, "Parámetros físicos")
print("Caso: {}".format(dirname))

In [ ]:
# Parámetros de la simulación (flopy.mf6.MFSimulation)
init_t = {
    'sim_name' : "trans",
}

# Parámetros para el tiempo (flopy.mf6.ModflowTdis)
# El mismo tdis de flow

# Parámetros para la solución numérica (flopy.mf6.ModflowIms)
# OJO: definiremos el modelo de solución más adelante
#ims_t = {
#    'linear_acceleration': "bicgstab"
#}

# Parámetros para el modelo de transporte (flopy.mf6.ModflowGwt)
gwt = { 
    'modelname': init_t["sim_name"],
    'save_flows': True
}

# Parámetros para la discretización espacial (flopy.mf6.ModflowGwtdis)
# El mismo dis de flow

# Parámetros para las condiciones iniciales (flopy.mf6.ModflowGwtic)
nlay, nrow, ncol = 1, 1, 30 #120
conc_inicial = np.loadtxt("conc_i.m3d")
conc_inicial = conc_inicial.reshape((nlay, nrow, ncol))
#conc_inicial = np.zeros((nlay, nrow, ncol))
#conc_inicial[0, 0, 11] = 10.0

ic_t = {
    'strt': conc_inicial #0.0
}

# Parámetros para MST (flopy.mf6.ModflowGwtmst)
mst = get_mst_dict(phys)

# Parámetros para ADV (flopy.mf6.ModflowGwtadv)
adv = {
    "scheme" : "TVD"
}

# Parámetros para DSP (flopy.mf6.ModflowGwtdsp)
dsp = {
    "xt3d_off" : True,
    "alh" : phys["longitudinal_dispersivity"],
    "ath1" : phys["longitudinal_dispersivity"],
}

# Parámetros para FMI (flopy.mf6.ModflowGwtfmi)
# OJO: no requerimos este paquete para leer la información del flujo de archivos.
#fmi = {
#    "packagedata" : [("GWFHEAD", f"{init['sim_name']}.hds", None),
#                     ("GWFBUDGET", f"{init['sim_name']}.bud", None),
#                    ]
#}

# Parámetros para SSM (flopy.mf6.ModflowGwtssm)
ssm = {
    "sources" : [["WEL-1", "AUX", "CONCENTRATION"]]
}

# Parámetros para OBS (flopy.mf6.ModflowGwtobs)
obs = {
    "digits" : 10, 
    "print_input" : True, 
    "continuous" : {
        "transporte.obs.csv": [
            ("X005", "CONCENTRATION", (0, 0, 0)),
            ("X405", "CONCENTRATION", (0, 0, 10)),
            ("X1105", "CONCENTRATION", (0, 0, 20)),
        ],
    }
}

# Parámetros para almacenar y mostrar la salida de la simulación (flopy.mf6.ModflowGwtoc)
oc_t = {
    'budget_filerecord': f"{init_t['sim_name']}.cbc",
    'concentration_filerecord': f"{init_t['sim_name']}.ucn",
    'saverecord' : [("CONCENTRATION", "ALL"), ("BUDGET", "LAST")],
    'printrecord' : [("CONCENTRATION", "LAST"), ("BUDGET", "LAST")],
}

# --- Inicialización de la simulación ---
# OJO: solo se definen los paquetes de GWT
o_gwt, packagest = xmf6.gwt.set_packages(o_sim, silent = True,
                                        gwt = gwt, 
                                        dis = dis, ic = ic_t, 
                                        mst = mst, adv = adv, dsp = dsp, ssm = ssm, 
                                        oc = oc_t)

# Definición del modelo de solución. Se realiza en este punto porque primero
# se requiere definir el objeto de flujo 'o_gwt' para conocer el nombre y
# posteriormente hacer el "registro" del modelo de solución en el objeto 'o_sim'.
o_ims_t = flopy.mf6.ModflowIms(
        o_sim,
        print_option="ALL",
        under_relaxation="NONE",
        linear_acceleration="BICGSTAB",
        scaling_method="NONE",
        reordering_method="NONE",
        filename=f"{o_gwt.name}.ims",
)
o_sim.register_ims_package(o_ims_t, [o_gwt.name])

# Se definen puntos de observación para la simulación de transporte
o_obs = xmf6.common.set_obs(o_gwt, obs, silent = True)

# Se agrega la componente del intercambio a la simulación.
# Ya debe estar definida la simulación con sus modelos numéricos,
# en este caso estos modelos están definidos en los objetos 'o_gwf' y 'o_gwt' 
flopy.mf6.ModflowGwfgwt(
    o_sim, exgtype="GWF6-GWT6", exgmnamea=o_gwf.name, exgmnameb=o_gwt.name
)

In [ ]:
# --- Escritura de archivos ---
o_sim.write_simulation(silent = True)

In [ ]:
# --- Ejecución de la simulación ---
o_sim.run_simulation(silent = False)

## Paso 4. Análisis de resultados.

In [ ]:
# --- Recuperamos los resultados de flujo de la simulación ---
# TODO: checar si se puede recuperar la información del objeto o_gwf directamente.
head = xmf6.gwf.get_head(o_gwf)
qx, qy, qz, n_q = xmf6.gwf.get_specific_discharge(o_gwf, text="DATA-SPDIS")


# --- Recuperamos los resultados de la concentración de la simulación ---
cobj = o_gwt.output.concentration()
#times = cobj.get_times()
#times = np.array(times)

# Obtener los tiempos realmente disponibles en el archivo
times = np.array(cobj.get_times())

# --- Recuperamos las coordenadas del dominio
x, y, z = o_gwf.modelgrid.xyzcellcenters
row_length = o_gwf.modelgrid.extent[1]

#--- Leemos la solución analítica de la ec. de transporte ---
"""
sol_path = 'analytic/' + dirname
a1_0 = np.load(sol_path + '/a1_x_0.npy')
a1_1 = np.load(sol_path + '/a1_x_1.npy')
a1_2 = np.load(sol_path + '/a1_x_2.npy')
"""

# --- Definición de la figura ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize =(6,6), height_ratios=[0.1,1,2])

# --- Gráfica 1. Malla ---
pmv = flopy.plot.PlotMapView(o_gwf, ax=ax1)
pmv.plot_grid(colors='dimgray', lw=0.5)
ax1.set_yticks(ticks=[0, 0.1])#, fontsize=8)
ax1.set_title("Mesh")

# --- Gráfica 2. Carga hidráulica---
ax2.plot(x[0], head[0, 0], marker=".", ls ="-", mec="blue", mfc="none", markersize="1", label = 'Head')
ax2.set_xlim(0, 2)
ax2.set_xticks(ticks=np.linspace(0, row_length,13))
#ax2.set_xlabel("Distance (cm)")
ax2.set_ylabel("Head")
ax2.grid()

# --- Gráfica 3. Concentración ---
# Solución analítica
"""
ax3.plot(x[0], a1_0, c = 'k', label='Analytic')
ax3.plot(x[0], a1_1, c = 'k')
ax3.plot(x[0], a1_2, c = 'k')
"""
# Solución numérica
ctimes = [1] #[1.0, 10.0, 20.0, 40.0] # tiempos a graficar
iskip = 1

for t in ctimes:
    conc = cobj.get_data(totim=t)

    x_plot = x[0][::iskip]
    c_plot = conc[0, 0][::iskip]

    line = ax3.plot(x_plot, c_plot, lw=1.5, alpha=0.9, label=f"GWT. Time = {t}")
    color = line[0].get_color()
    ax3.scatter(x_plot, c_plot, color=color, edgecolor="k", s=25, zorder=5)

# --- Ajustar límites automáticamente ---
ax3.relim()        # Recalcula límites con todos los artistas
ax3.autoscale()    # Ajusta el eje X y Y automáticamente

# Decoración adicional
ax3.grid(True)
ax3.set_xlabel("Distance (m)")
ax3.set_ylabel("Concentration")
ax3.legend(fontsize=9) 

plt.tight_layout()
plt.show()

# Referencias

[1] MODFLOW 6 – Example problems, MODFLOW 6 Development Team, with contributions from Chieh Ying Chen and Mike Toews 02/07/2024.**  **32 MOC3D Problem 1**. (Archivo: `mf6examples.pdf` del directorio `doc` de la distribución de MODFLOW 6).

[2] Langevin, C. D., Hughes, J. D., Provost, A. M., Russcher, M. J., & Panday, S. (2023). MODFLOW as a configurable Multi‐Model Hydrologic Simulator. Ground Water. https://doi.org/10.1111/gwat.13351.
